# Restroom Icon Matching

In [1]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00


In [22]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F
import open_clip

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [23]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)

In [35]:
model_name = 'ViT-H-14'
pretrained = 'laion2b_s32b_b79k'
device = 'cuda'

In [36]:
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
model = model.to(device)
tokenizer = open_clip.get_tokenizer(model_name)
model.eval()

open_clip_model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [37]:
def infer(img_paths):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    embeddings = []
    for path in tqdm(img_paths):
        img = Image.open(path)
        x = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad(), torch.cuda.amp.autocast() :
            emb = model.encode_image(x)
            
        emb = F.normalize(emb, p=2, dim=-1)
        embeddings.append(emb)

    embeddings = torch.cat(embeddings)

    return embeddings.cpu().detach().numpy()

In [38]:
from sklearn.neighbors import NearestNeighbors

def match_images(BASE_DATA_DIR):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then return the match indices as a list.
    """
    QUERY_DIR = BASE_DATA_DIR / "query"
    NON_QUERY_DIR = BASE_DATA_DIR / "gallery"

    query_image_paths = list(QUERY_DIR.glob("*.png"))
    non_query_image_paths = list(NON_QUERY_DIR.glob("*.png"))

    query_image_paths_str = [str(p) for p in query_image_paths]
    non_query_image_paths_str = [str(p) for p in non_query_image_paths]

    sort_paths_by_number(query_image_paths_str)
    sort_paths_by_number(non_query_image_paths_str)

    query_embeddings = infer(query_image_paths_str)
    non_query_embeddings = infer(non_query_image_paths_str)

    knn = NearestNeighbors(n_neighbors=2, metric='cosine') 
    knn.fit(non_query_embeddings)
    idxs = knn.kneighbors(query_embeddings, n_neighbors=2, return_distance=False)
    
    results = []
    for i in range(len(idxs)) : 
        results.append(idxs[i][1] + 1)
    return results

In [39]:
"""Generate submission CSV for test set only"""
DATA_PATH = Path("/kaggle/input/competitions/restroom-ioai-2025/test_set")

# Process test set and get results
print("Processing test set...")
test_results = match_images(DATA_PATH / "test_set")

# Create CSV in the required format
test_results = [int(x) for x in test_results]
submission_df = pd.DataFrame({
    'id': [0],
    'array': [test_results]
})

# Save to CSV
output_file = "Restroom Baseline.csv"
submission_df.to_csv(output_file, index=False)

print(f"Submission file created successfully at {output_file}")
print(f"Results shape: {len(test_results)} matches")

Processing test set...


  0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipykernel_55/2731000109.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast() :
100%|██████████| 80/80 [00:03<00:00, 21.46it/s]

Submission file created successfully at Restroom Baseline.csv
Results shape: 40 matches


In [40]:
submission_df['array'][0]

[37,
 13,
 45,
 17,
 58,
 49,
 14,
 59,
 39,
 7,
 18,
 21,
 44,
 78,
 22,
 19,
 57,
 35,
 27,
 46,
 50,
 12,
 16,
 4,
 54,
 31,
 20,
 29,
 15,
 37,
 77,
 78,
 74,
 79,
 63,
 65,
 9,
 71,
 73,
 56]